### Getting started with gridfm-graphkit toolkit
<div>
<img src="/dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/docs/figs/KIT.png" width="100" align="right" />
</div>

This notebook serves as a guide to the [gridfm-graphkit](https://github.com/gridfm/gridfm-graphkit) functionality using data generated through [gridfm-datakit](https://github.com/gridfm/gridfm-datakit)



[Gridfm-graphkit](https://github.com/gridfm/gridfm-datakit) is a toolkit designed for Foundation Model development for power grid applications. 
<br>
<br>
**Functionality:**
- Multi-grid training
- Generalizable representations
- Physics-informed loss
- Multiple tasks in one model:
    - Power Flow (PF)
    - Optimal Power Flow (OPF)
    - State Estimation

#### Environment
Create and activate a virtual environment (make sure you use the right python version = 3.10, 3.11 or 3.12. I highly recommend 3.12. Follow instructions on [README.md](../../README.md)

### 1. Power Flow (PF): 
Estimate line flows & bus voltages for electric grids

#### 1.1. Training
- Run model training on synthetic grid data generated from using [gridfm-datakit](https://github.com/gridfm/gridfm-datakit), or other grid data
- Data format for model training: heterogeneous graphs, scenario-batched parquet files

<div>
<img src="/dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/docs/figs/Training2.png" width="1000"/>
</div>


#### For example, let's consider training a model for case14_ieee data specific to the PF task

- Edit [config](../config/HGNS_PF_datakit_case14.yaml) file to set model training parameters

<br>

**Warning: The below cell may require a GPU to run, depending on the dataset size, ensure you have sufficient resource allocation**

In [2]:
# Run training
!gridfm_graphkit train --config ../config/HGNS_PF_datakit_case14.yaml \
--exp_name tut_case14 --run_name training_case14 --log_dir ../mlflows/training/ \
--data_path ../data/pf/

/dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | '

Check mlflow artifacts in ```log_dir``` to assess the error metrics

In [ ]:
!mlflow ui --port 2505 --backend-store-uri /dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/examples/mlflows/training/

#### 1.2. Fine-tuning
Fine-tune an existing pre-trained model using a smaller labeled dataset.

Finetuning datasets for PF and OPF generated from [gridfm-datakit](https://github.com/gridfm/gridfm-datakit) are available for download on GridFM HuggingFace # DO OTHERS HAVE ACCESS?

In [ ]:
# Run fine-tuning
!gridfm_graphkit finetune --config ../config/HGNS_PF_datakit_case14.yaml \
--model_path ../mlflows/training/242437082336001572/f7a9fa11a3534b11b17b730c3fea9da0/artifacts/model/best_model_state_dict.pt \
--exp_name tut_case14_ft --run_name ft_case14 \
--log_dir ../mlflows/fine_tuning/ --data_path ../data/pf/

Check mlflow artifacts

In [ ]:
!mlflow ui --port 2505 --backend-store-uri ../mlflows/fine_tuning/

#### 1.3. Evaluate
Evaluate the model's perfromance on a labeled test set dataset:

In [5]:
# Run evaluation
!gridfm_graphkit evaluate --config ../config/HGNS_PF_datakit_case14.yaml \
--model_path ../mlflows/training/242437082336001572/f7a9fa11a3534b11b17b730c3fea9da0/artifacts/model/best_model_state_dict.pt \
--exp_name tut_case14_eval --run_name eval_case14 \
--log_dir ../mlflows/eval/ --data_path ../data/pf/ \
--normalizer_stats ../mlflows/training/242437082336001572/f7a9fa11a3534b11b17b730c3fea9da0/artifacts/stats/normalizer_stats.pt

/dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Seed set to 0
Loading model weights from ../mlflows/training/242437082336001572/f7a9fa11a3534b11b17b730c3fea9da0/artifacts/model/best_model_state_dict.pt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (cuda)

In [ ]:
!mlflow ui --port 2505 --backend-store-uri /dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/examples/mlflows/eval/

#### Outputs
Evaluations metrics are saved in: ```../mlflows/eval/exp_name/run_name/artifacts/test/```

#### 1.4. Predict
Use gridfm-graphkit to run predictions on unseen data. 

Let's try predictions for case30_ieee - This model is trained on case30_ieee data

In [7]:
# Run predictions
!gridfm_graphkit predict --config ../config/HGNS_PF_datakit_case30.yaml \
--model_path ../mlflows/training/697355889450429644/f34b111eb62e42738f2c7f446ba25419/artifacts/model/best_model_state_dict.pt \
--exp_name tut_case30_pred --run_name pred_case30 \
--log_dir ../mlflows/predict/ --data_path /dccstor/gridfm/powermodels_data/v4/evaluation_pretraining/pf/ \
--normalizer_stats ../mlflows/training/697355889450429644/f34b111eb62e42738f2c7f446ba25419/artifacts/stats/normalizer_stats.pt

/dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Seed set to 0
Loading model weights from ../mlflows/training/697355889450429644/f34b111eb62e42738f2c7f446ba25419/artifacts/model/best_model_state_dict.pt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (cuda)

In [9]:
# View predictions 
import pandas as pd
preds = pd.read_parquet("../data/pf/predictions/case30_ieee/case30_ieee_predictions.parquet")
preds.head()

,scenario,bus,pd_mw,qd_mvar,vm_pu_target,va_target,pg_mw_target,qg_mvar_target,is_pq,is_pv,is_ref,vm_pu,va,pg_mw,qg_mvar,active res. (MW),reactive res. (MVar),PBE
0,1711,0,0.000000,0.000000,1.060000,0.000000,214.619614,-0.346012,0,0,1,1.059653,0.000000,214.331741,-1.072397,0.000061,0.000067,0.000091
1,1711,1,17.500034,11.703244,1.035024,-0.076370,28.214033,11.088933,0,1,0,1.035024,-0.076400,28.214033,11.687276,0.000605,0.000321,0.000685
2,1711,2,1.866887,0.777449,1.030730,-0.095942,0.000000,0.000000,1,0,0,1.030636,-0.095903,0.000000,0.000000,0.000387,0.008692,0.008701
3,1711,3,5.665118,1.008988,1.024565,-0.120052,0.000000,0.000000,1,0,0,1.024530,-0.119995,0.000000,0.000000,0.001756,0.003821,0.004205
4,1711,4,83.824150,17.717951,0.997685,-0.190671,0.000000,15.564273,0,1,0,0.997685,-0.190628,0.000000,15.557042,0.002365,0.000080,0.002366


### 2. Optimal Power Flow (OPF) 

Directly map load scenarios to optimal generator setpoints subject to constraints

<div>
<img src="/dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/docs/figs/OPF.png" width="800"/>
</div>

- Edit [config](../config/HGNS_OPF_datakit_case14.yaml) file to set model parameters for OPF
- Update ```task_name: OptimalPowerFlow``` and add ```losses```, ```loss_weights```
<br>
<br>
You are now able to use gridfm-graphkit to run model training, fine-tuning, evaluation and prediction 

### 3. State Estimation

Infer actual system states from noisy sensors or unknown grid configurations

<div>
<img src="/dccstor/gridfm/tamara/modelling/updates/new_eval/gridfm-graphkit/docs/figs/SE.png" width="300"/>
</div>


- Edit [config](../config/HGNS_SE_datakit_case14.yaml) file to set model parameters for SE
- Ensure ```task_name: StateEstimation``` and update ```task``` parameters for SE, based on template provided. 
<br>
<br>
You are now able to use gridfm-graphkit to run model training, fine-tuning, evaluation and prediction 